Training PaddleOCR on CPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import cv2

In [ ]:
import os

# === CHANGE THIS to your actual folder in Google Drive ===
DRIVE_DATASET_PATH = '/content/drive/MyDrive/my_dataset'

# These are the subfolders you created earlier
TRAIN_DIR = os.path.join(DRIVE_DATASET_PATH, 'train')
VAL_DIR   = os.path.join(DRIVE_DATASET_PATH, 'val')

# Where to save your trained model (also in Drive so it survives session resets)
OUTPUT_DIR = '/content/drive/MyDrive/paddleocr_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify your dataset is visible
print("Train images found:", len(os.listdir(TRAIN_DIR)))
print("Val images found:",   len(os.listdir(VAL_DIR)))

Train images found: 921
Val images found: 176


In [ ]:
import os

PRETRAIN_DIR = '/content/pretrain_models'
os.makedirs(PRETRAIN_DIR, exist_ok=True)

%cd {PRETRAIN_DIR}

if not os.path.exists('en_PP-OCRv3_rec_train'):
    !wget -q https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_rec_train.tar
    !tar -xf en_PP-OCRv3_rec_train.tar
    print("Downloaded and extracted!")
else:
    print("Already downloaded.")

PRETRAIN_MODEL = os.path.join(PRETRAIN_DIR, 'en_PP-OCRv3_rec_train/best_accuracy')

/content/pretrain_models
Downloaded and extracted!


In [ ]:
import os
from IPython.display import display, clear_output, Image
import ipywidgets as widgets

# ── change this to your folder ──────────────────────────────────────────────
IMAGE_DIR  = '/content/drive/MyDrive/my_dataset/val'
LABEL_FILE = os.path.join(IMAGE_DIR, 'Label.txt')
# ────────────────────────────────────────────────────────────────────────────

# Load already-labeled so you can resume mid-session
labeled = {}
if os.path.exists(LABEL_FILE):
    with open(LABEL_FILE, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) == 2:
                labeled[parts[0]] = parts[1]

all_images = sorted([f for f in os.listdir(IMAGE_DIR)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
to_label = [f for f in all_images if f not in labeled]

print(f"Total images : {len(all_images)}")
print(f"Already done : {len(labeled)}")
print(f"Remaining    : {len(to_label)}")

Total images : 175
Already done : 72
Remaining    : 103


In [ ]:
import os
from IPython.display import display, clear_output, Image
import ipywidgets as widgets

# ── change this to your folder ───────────────────────────────────────────────
IMAGE_DIR  = '/content/drive/MyDrive/my_dataset/val'
LABEL_FILE = os.path.join(IMAGE_DIR, 'Label.txt')
# ─────────────────────────────────────────────────────────────────────────────

labeled = {}
if os.path.exists(LABEL_FILE):
    with open(LABEL_FILE, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) == 2:
                labeled[parts[0]] = parts[1]

all_images = sorted([f for f in os.listdir(IMAGE_DIR)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
to_label = [f for f in all_images if f not in labeled]

print(f"Total images : {len(all_images)}")
print(f"Already done : {len(labeled)}")
print(f"Remaining    : {len(to_label)}")

idx          = [0]
label_file_h = open(LABEL_FILE, 'a', encoding='utf-8')

progress  = widgets.IntProgress(value=0, min=0, max=len(to_label),
                                 description='Progress:', bar_style='info',
                                 layout=widgets.Layout(width='100%'))
counter   = widgets.HTML()
img_out   = widgets.Output(layout=widgets.Layout(border='1px solid #ccc',
                                                  min_height='200px'))
txt       = widgets.Text(placeholder='Type transcription then press Enter',
                         layout=widgets.Layout(width='80%'))
skip_btn  = widgets.Button(description='Skip (no text)',
                            button_style='warning',
                            layout=widgets.Layout(width='18%'))
status    = widgets.HTML()

ui = widgets.VBox([progress, counter, img_out,
                   widgets.HBox([txt, skip_btn]), status])
display(ui)

def load_image(i):
    if i >= len(to_label):
        counter.value     = '<b>All done!</b>'
        status.value      = '<span style="color:green">✓ All images labeled. Label.txt saved to Drive.</span>'
        txt.disabled      = True
        skip_btn.disabled = True
        label_file_h.close()
        return
    fname = to_label[i]
    counter.value  = (f'<b>{i+1} / {len(to_label)}</b> &nbsp;—&nbsp; '
                      f'<code>{fname}</code>')
    progress.value = i
    with img_out:
        clear_output(wait=True)
        display(Image(filename=os.path.join(IMAGE_DIR, fname), width=640))
    txt.value = ''

def save(transcription):
    fname = to_label[idx[0]]
    if transcription.strip():
        label_file_h.write(f"{fname}\t{transcription.strip()}\n")
        label_file_h.flush()
        status.value = f'<span style="color:green">Saved: {transcription.strip()}</span>'
    else:
        status.value = f'<span style="color:orange">Skipped: {fname}</span>'
    idx[0] += 1
    load_image(idx[0])

txt.on_submit(lambda w: save(w.value))
skip_btn.on_click(lambda b: save(''))

load_image(0)

Total images : 175
Already done : 72
Remaining    : 103


In [ ]:
import os
import ipywidgets as widgets
from IPython.display import display, clear_output, Image

# ── same path as before ──────────────────────────────────────────────────────
IMAGE_DIR  = '/content/drive/MyDrive/my_dataset/train'
LABEL_FILE = os.path.join(IMAGE_DIR, 'Label.txt')
# ─────────────────────────────────────────────────────────────────────────────

# Load all labels
entries = []
with open(LABEL_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            entries.append(list(parts))  # [filename, transcription]

print(f"Loaded {len(entries)} labeled entries.")

# ── UI ───────────────────────────────────────────────────────────────────────
idx       = [0]
img_out   = widgets.Output(layout=widgets.Layout(min_height='200px',
                                                  border='1px solid #ccc'))
counter   = widgets.HTML()
txt       = widgets.Text(layout=widgets.Layout(width='80%'))
save_btn  = widgets.Button(description='Save correction',
                            button_style='success',
                            layout=widgets.Layout(width='18%'))
prev_btn  = widgets.Button(description='◀ Prev', layout=widgets.Layout(width='15%'))
next_btn  = widgets.Button(description='Next ▶', layout=widgets.Layout(width='15%'))
status    = widgets.HTML()
jump_num  = widgets.BoundedIntText(value=1, min=1, max=len(entries),
                                    description='Jump to:',
                                    layout=widgets.Layout(width='200px'))
jump_btn  = widgets.Button(description='Go', layout=widgets.Layout(width='60px'))

ui = widgets.VBox([
    counter, img_out, txt,
    widgets.HBox([save_btn]),
    widgets.HBox([prev_btn, next_btn]),
    widgets.HBox([jump_num, jump_btn]),
    status
])
display(ui)

def show(i):
    idx[0] = max(0, min(i, len(entries) - 1))
    fname, transcription = entries[idx[0]]
    counter.value = (f'<b>{idx[0]+1} / {len(entries)}</b> &nbsp;—&nbsp; '
                     f'<code>{fname}</code>')
    txt.value = transcription
    with img_out:
        clear_output(wait=True)
        fpath = os.path.join(IMAGE_DIR, fname)
        if os.path.exists(fpath):
            display(Image(filename=fpath, width=640))
        else:
            print(f"Image not found: {fpath}")

def save_correction(btn=None):
    entries[idx[0]][1] = txt.value.strip()
    # Rewrite the entire Label.txt with corrections
    with open(LABEL_FILE, 'w', encoding='utf-8') as f:
        for fname, text in entries:
            f.write(f"{fname}\t{text}\n")
    status.value = (f'<span style="color:green">✓ Correction saved: '
                    f'<b>{txt.value.strip()}</b></span>')

save_btn.on_click(save_correction)
txt.on_submit(lambda w: save_correction())   # Enter also saves
prev_btn.on_click(lambda b: show(idx[0] - 1))
next_btn.on_click(lambda b: show(idx[0] + 1))
jump_btn.on_click(lambda b: show(jump_num.value - 1))

show(0)

Loaded 877 labeled entries.


In [ ]:
!python -m pip install paddlepaddle==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/ -q

In [ ]:
!pip install paddleocr -q

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
import paddle
print(paddle.__version__)

In [ ]:
import paddleocr
print(paddleocr.__version__)

In [ ]:
from paddleocr import TextRecognition

model= TextRecognition(model_name="PP-OCRv5_mobile_rec", device="cpu")


In [ ]:
output = model.predict(
    input="/media/test5.jpg",
    batch_size=1
)

In [ ]:
for res in output:
    res.print()

In [ ]:
for res in output:
    res.print()


In [ ]:
# Cell 1
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/paddleocr_finetune', exist_ok=True)
%cd /content/drive/MyDrive/paddleocr_finetune

# Cell 2 — Install PaddlePaddle (CPU)
!python -m pip install paddlepaddle==3.0.0 \
    -i https://www.paddlepaddle.org.cn/packages/stable/cpu/ -q



# Cell 3 — Clone PaddleOCR source and install training dependencies
!git clone https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR
!pip install -r requirements.txt -q

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/paddleocr_finetune
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.3/192.9 MB 58.2 kB/s eta 0:55:07
ERROR: Operation cancelled by user
fatal: destination path 'PaddleOCR' already exists and is not an empty directory.
/content/drive/MyDrive/paddleocr_finetune/PaddleOCR
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 107.2 MB/s eta 0:00:00


In [ ]:
# Cell 9 — Auto-generate dict.txt from your labels
with open('/content/drive/MyDrive/my_dataset/train/Label.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

# Collect every unique character from all labels
all_chars = set()
for line in lines:
    parts = line.strip().split('\t')
    if len(parts) == 2:
        all_chars.update(parts[1])

# Sort and write, one character per line
sorted_chars = sorted(all_chars)
with open('/content/drive/MyDrive/my_dataset/train/Dict.txt', 'w', encoding='utf-8') as f:
    for ch in sorted_chars:
        f.write(ch + '\n')

print(f"Dictionary has {len(sorted_chars)} unique characters.")
print("Sample chars:", sorted_chars[:20])

In [ ]:
!mkdir -p pretrained_models
!wget -q https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_mobile_rec_pretrained.pdparams \
    -O pretrained_models/PP-OCRv5_mobile_rec_pretrained.pdparams

!ls -lh pretrained_models/

In [ ]:
# Cell 11 — Copy the base config
!mkdir -p data  # Create the 'data' directory if it doesn't exist
!cp configs/rec/PP-OCRv5/PP-OCRv5_mobile_rec.yml data/finetune_config.yml

In [ ]:
import numpy as np


In [ ]:
img_test= '/media/test.jpg'
full_preprocess(img_test, '/media/test_processed.jpg')

In [ ]:
def full_preprocess(img_path, save_path):
    img = cv2.imread(img_path)
    if img is None:
        print(f"Could not read: {img_path}")
        return

    # 1. Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 2. CLAHE contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)

    # 3. Otsu binarization
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 4. Deskew
    coords = np.column_stack(np.where(binary < 128)) # dark pixels = text
    if len(coords) >= 10:
        angle = cv2.minAreaRect(coords)[-1]
        if angle < -45:
            angle = -(90 + angle)
        else:
            angle = -angle
        if abs(angle) > 0.5:
            h, w = binary.shape
            center = (w // 2, h // 2)
            M = cv2.getRotationMatrix2D(center, angle, 1.0)
            binary = cv2.warpAffine(binary, M, (w, h),
                                    flags=cv2.INTER_CUBIC,
                                    borderMode=cv2.BORDER_REPLICATE)

    cv2.imwrite(save_path, binary)

In [ ]:


# --- Run on train and val ---
for split in ['train', 'val']:
    in_dir = f'/content/drive/MyDrive/my_dataset/{split}'
    out_dir = f'/content/drive/MyDrive/my_dataset_{split}_processed'
    os.makedirs(out_dir, exist_ok=True)

    count = 0
    for fname in os.listdir(in_dir):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            full_preprocess(
                os.path.join(in_dir, fname),
                os.path.join(out_dir, fname)
            )
            count += 1

    print(f"{split}: processed {count} images → {out_dir}")

In [ ]:
# Cell 12 — Patch the config (complete transforms rewrite)
import yaml

with open('data/finetune_config.yml', 'r') as f:
    config = yaml.safe_load(f)

# --- Global settings ---
config['Global']['use_gpu'] = False
config['Global']['epoch_num'] = 20
config['Global']['save_model_dir'] = './output/PP-OCRv5_mobile_rec_finetuned'
config['Global']['save_epoch_step'] = 10
config['Global']['eval_batch_step'] = [0, 20]
config['Global']['pretrained_model'] = './pretrained_models/PP-OCRv5_mobile_rec_pretrained.pdparams'
config['Global']['character_dict_path'] = '/content/drive/MyDrive/my_dataset/train/Dict.txt'
config['Global']['use_space_char'] = True
config['Global']['print_batch_step'] = 1

# Find max label length
with open('/content/drive/MyDrive/my_dataset/train/Label.txt') as f:
    lines = f.readlines()
max_len = max(len(l.strip().split('\t')[1]) for l in lines if '\t' in l)
config['Global']['max_text_length'] = max_len
print(f"Max text length set to: {max_len}")

DICT_PATH = '/content/drive/MyDrive/my_dataset/train/Dict.txt'

# --- Fully rewrite Train transforms ---
config['Train']['dataset']['transforms'] = [
    {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
    {'RecAug': None},
    {'MultiLabelEncode': {
        'gtc_encode': 'NRTRLabelEncode',
        'max_text_length': max_len,
        'character_dict_path': DICT_PATH,
        'use_space_char': True,
    }},
    {'RecResizeImg': {'image_shape': [3, 48, 320]}},
    {'KeepKeys': {'keep_keys': ['image', 'label_ctc', 'label_gtc', 'length']}},
]

# --- Fully rewrite Eval transforms ---
config['Eval']['dataset']['transforms'] = [
    {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
    {'MultiLabelEncode': {
        'gtc_encode': 'NRTRLabelEncode',
        'max_text_length': max_len,
        'character_dict_path': DICT_PATH,
        'use_space_char': True,
    }},
    {'RecResizeImg': {'image_shape': [3, 48, 320]}},
    {'KeepKeys': {'keep_keys': ['image', 'label_ctc', 'label_gtc', 'length']}},
]

# --- Train dataset ---
config['Train']['dataset']['name'] = 'SimpleDataSet'
config['Train']['dataset']['data_dir'] = '/content/drive/MyDrive/my_dataset_train_processed'
config['Train']['dataset']['label_file_list'] = ['/content/drive/MyDrive/my_dataset/train/Label.txt']
config['Train']['loader']['batch_size_per_card'] = 4
config['Train']['loader']['num_workers'] = 0
if 'sampler' in config['Train']:
    del config['Train']['sampler']

# --- Eval dataset ---
config['Eval']['dataset']['name'] = 'SimpleDataSet'
config['Eval']['dataset']['data_dir'] = '/content/drive/MyDrive/my_dataset_val_processed'
config['Eval']['dataset']['label_file_list'] = ['/content/drive/MyDrive/my_dataset/val/Label.txt']
config['Eval']['loader']['batch_size_per_card'] = 4
config['Eval']['loader']['num_workers'] = 0

with open('data/finetune_config.yml', 'w') as f:
    yaml.dump(config, f, allow_unicode=True, default_flow_style=False)

print("Config saved.")

# --- Verify ---
with open('data/finetune_config.yml', 'r') as f:
    saved = yaml.safe_load(f)

print("\n✅ Train transforms:")
for t in saved['Train']['dataset']['transforms']:
    print("  ", list(t.keys())[0], "→", list(t.values())[0])

print("\n✅ Eval transforms:")
for t in saved['Eval']['dataset']['transforms']:
    print("  ", list(t.keys())[0], "→", list(t.values())[0])


In [ ]:
# Verify the transform fix worked
with open('data/finetune_config.yml', 'r') as f:
    saved = yaml.safe_load(f)
train_transforms = saved['Train']['dataset'].get('transforms', [])
print("\nTrain transforms after fix:")
for t in train_transforms:
    print(" ", list(t.keys())[0] if isinstance(t, dict) else t)


In [ ]:
!rm -rf ./output/PP-OCRv5_mobile_rec_finetuned

In [ ]:
# Inspect what transforms are still in the saved config
import yaml
with open('data/finetune_config.yml') as f:
    c = yaml.safe_load(f)
for t in c['Train']['dataset'].get('transforms', []):
    print(t)


In [ ]:
# Cell 13 — Run training
!python tools/train.py -c data/finetune_config.yml

In [ ]:
# Cell 14 — Evaluate on the validation set
!python tools/eval.py \
    -c data/finetune_config.yml \
    -o Global.pretrained_model=output/PP-OCRv5_mobile_rec_finetuned/best_accuracy.pdparams

In [ ]:
# Cell 15 — Export to inference format
!python tools/export_model.py \
    -c data/finetune_config.yml \
    -o Global.pretrained_model=output/PP-OCRv5_mobile_rec_finetuned/best_accuracy.pdparams \
       Global.save_inference_dir="output/my_handwriting_model"

!ls output/my_handwriting_model/

In [ ]:
# Cell 18 — Back up to Drive so you don't lose it
!cp -r output/my_handwriting_model \
    /content/drive/MyDrive/paddleocr_finetune/my_handwriting_model

print("Model saved to Google Drive.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Step 1 — Mount Drive and install dependencies (run if in a fresh session)


!python -m pip install paddlepaddle==3.0.0 \
    -i https://www.paddlepaddle.org.cn/packages/stable/cpu/ -q
!pip install paddleocr -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.9/192.9 MB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.1/88.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/

In [ ]:
# Step 2 — Restart runtime after install, then run this cell
import os
os.kill(os.getpid(), 9)

In [ ]:
# Step 4 — Run inference with your fine-tuned model
from paddleocr import TextRecognition

MODEL_DIR = '/content/drive/MyDrive/paddleocr_finetune/my_handwriting_model'



Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [ ]:
test_image_path= "/media/test5.jpg"

In [ ]:
model = TextRecognition(
    model_name="PP-OCRv5_mobile_rec",
    model_dir=MODEL_DIR,
    device="cpu"
)

output = model.predict(input=test_image_path, batch_size=1)

for res in output:
    res.print()